In [13]:
workspace_id = ""
destination_lakehouse_id = ""
destination_lakehouse_path = ""
deployment_environment = ""

StatementMeta(, 9dfa215f-5d98-43a2-80b9-6b00a1402949, 20, Finished, Available, Finished)

In [18]:
from pyspark.sql.functions import input_file_name, col, count, countDistinct, coalesce, lit
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType
from concurrent.futures import ThreadPoolExecutor
import unittest
from pyspark.sql import SparkSession
import io
import logging
import sempy.fabric as fabric
import xmlrunner
from delta import DeltaTable
from typing import Optional, List, Tuple, Dict
from pyspark.sql.utils import AnalysisException

EXPECTED_TABLE_COUNTS = {
    "care_site": 15582,
    "cdm_source": 0,
    "cohort": 0,
    "cohort_definition": 0,
    "concept": 8623774,
    "concept_ancestor": 70186504,
    "concept_class": 416,
    "concept_relationship": 59374292,
    "concept_synonym": 2051247,
    "condition_era": 0,
    "condition_occurrence": 1973141,
    "cost": 0,
    "death": 5768,
    "device_exposure": 0,
    "domain": 0,
    "dose_era": 0,
    "drug_era": 0,
    "drug_exposure": 2675259,
    "drug_strength": 0,
    "episode": 0,
    "episode_event": 0,
    "fact_relationship": 0,
    "fhir_system_to_omop_vocab_mapping": 13,
    "image_occurrence": 0,
    "location": 62803,
    "measurement": 12375682,
    "metadata": 0,
    "note": 12814,
    "note_nlp": 0,
    "observation": 7339007,
    "observation_period": 0,
    "payer_plan_period": 0,
    "person": 47564,
    "procedure_occurrence": 5299006,
    "provider": 15614,
    "relationship": 0,
    "source_to_concept_map": 0,
    "specimen": 0,
    "visit_detail": 0,
    "visit_occurrence": 3194934,
    "vocabulary": 127
}

class OmopDataValidationTests(unittest.TestCase):

    def __init__(self, methodName='runTest', spark=None, workspace_id = None, bronze_lakehouse_id = None, databases = []):
        super().__init__(methodName)
        logging.basicConfig()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.workspace_id = workspace_id
        self.bronze_lakehouse_id = bronze_lakehouse_id
        self.databases = databases

    def validate_df_cols_standard_nonstandard(self, df: DataFrame, col_name: Optional[int], expected: int, isStandard: bool = True)-> None:
        """
        Verifies standard concepts are hydrated with non 0 concept ids and non standard concept are 0
        Arguments:
            df: DataFrame - DataFrame to validate
            col_name: str - Column name to validate
            expected: int - Expected count of non 0 concept ids
            isStandard: bool - True if standard concept, False if non standard concept
        Returns:
            None, raises ValidationFailureException if validation fails
        """
        if isStandard:
            col_count = df.filter(col('standard_concept')=='S').collect()[0].__getitem__(col_name)
            self.assertGreater(col_count, expected)
        if not isStandard:
            self.assertEqual(df.filter(col('standard_concept') != 'S').count(), expected)

    def validate_df(self, df: DataFrame, 
                columns: Optional[List[str]] = None, 
                n_cols: Optional[int] = None, 
                check_null_values: Optional[List[str]] = None,
                ) -> Tuple[bool, str]:

        """
        Validates a DataFrame based on specified criteria.

        Parameters:
        - df (pd.DataFrame): The DataFrame to be validated.
        - columns (list, optional): List of column names that should be present in the DataFrame.
        - n_cols (int, optional): Number of expected columns in the DataFrame.
        - check_null_values (bool, optional): Check for the presence of null values in the DataFrame.

        Returns:
        - tuple: (bool, str) indicating success or failure, and an optional description of the problem.
        """

        # Validate number of columns
        if n_cols is not None and len(df.columns) != n_cols:
            error_message = f"Error: Expected {n_cols}columns, but found {len(df.columns)}columns."
            raise self.createValidationFailureException(error_message)

        # Validate columns
        if columns is not None:
                if not set(columns).issubset(df.columns):
                    missing_columns = set(columns) - set(df.columns)
                    error_message = f"Error: Missing columns: {missing_columns}."
                    raise self.createValidationFailureException(error_message) 

        # Validate null values  ONLY SPECIFIC COLUMNS
        for value in check_null_values:
            if value and df.filter(df[value].isNull()).count() != 0:
                error_message = "DataFrame contains null values."
                raise  ValidationFailureException(error_message)
        return (True, "Dataframe has passed all the validations")

    def validate_df_col_count(self, actual: int, expected: int)-> None:

        if actual <= expected:
            raise self.createValidationFailureException('Value is lower than the target expected')
    
    def validate_df_col_range(self, actual: int, desired_range: Tuple[int,int])-> None:

        if actual<= desired_range[0] or actual >= desired_range[1]:
            raise self.createValidationFailureException('Provided value is out of bounds')

    def verify_transformed_data_quality(self, silver_omop_config: dict, silver_lakehouse: str, assert_dfs: bool)-> None:
        """Verfies the omop transformed data is ingested completely

        Arguments:
            silver_omop_config: dict - A dictionary of silver_omop_config: fhir source table name, source table filtering condition, omop target table name and expected count
            silver_lakehouse: str - Silver lakehouse name
            assert_dfs: flag if true checks both the dfs have equal row counts

        Returns:
            None, raises ValidationFailureException if the row counts does not match for the silver and omop tables or AnalysisException if any of the SQL queries fail
        """
        for key, value in silver_omop_config.items():
            source_table = key
            for (omop_target_table, condition, expected_count) in value:   
                # Check if expected_count is None only when assert_dfs is False
                if not assert_dfs and expected_count is None:
                    raise self.createValidationFailureException(f"Invalid configuration: `expected_count` is None for the table {source_table} -> {omop_target_table} when `assert_dfs` is False.")
                try:
                    if condition:        
                        silver_df = spark.sql(f"SELECT * FROM {silver_lakehouse}.{source_table}").filter(condition)
                    else:
                        silver_df = spark.sql(f"SELECT * FROM {silver_lakehouse}.{source_table}")

                    omop_df = spark.sql(f"SELECT * FROM healthcare1_msft_gold_omop.{omop_target_table}")
                    source_count = silver_df.count()
                    target_count = omop_df.count()

                    if assert_dfs:
                        self.assertEqual(source_count, target_count)
                    elif expected_count: 
                        source_count = expected_count
                        self.assertEqual(source_count, target_count)
                except AnalysisException as e:
                    raise AnalysisException(f"SQL query failed for table {source_table} or {omop_target_table}. Details: {str(e)}")
                except AssertionError as e:
                    raise self.createValidationFailureException(f"Silver and OMOP lakehouse row counts does not match for the tables: {source_table}-{source_count}<->{omop_target_table}-{target_count}")

    def get_workspace_items_by_type(self, workspace_id, item_type):
        client = fabric.FabricRestClient()

        response = client.get(f"v1/workspaces/{workspace_id}/items?type={item_type}")
        items = response.json()['value']

        item_dict = {}
        for item in items:
            item_dict[item['displayName']] = item
        
        return item_dict

    def omop_concept_validation(self, omop_tables_concept_pairs: Dict[str, list[int]])-> None:
        """Test verifies that 100% of all the sample data shipped have the associated concept codes and concept_ids populated in OMOP.

        Parameters:
            - df (pd.DataFrame): The DataFrame to be validated.
            - omop_tables_concept_pairs: This is a dictionary where:
                - Keys (str): Each key represents the table name in OMOP lakehouse.
                - Values (list[int]): Each value is a list of integers representing concept IDs associated with the corresponding OMOP table.

        Returns:
            None, raises ValidationFailureException if any of the concept_ids are not mapped resulting in null values
        """
        for table, concepts in omop_tables_concept_pairs.items():
            concept_id_cols = ',' .join(f"{concept}" for concept in concepts)
            print(concept_id_cols)
            query = f"SELECT {concept_id_cols} FROM healthcare1_msft_gold_omop.{table};"
            df = spark.sql(query)
            if df.isEmpty():
                continue
            for concept in concepts:
                concept_id_count = df.filter(col(concept) != 0).count() # when the mapping is corrupted the concept_ids will default to 0
                print(f"Column '{concept}': in {table} Count is : {concept_id_count}")
                self.assertNotEqual(concept_id_count, 0)

    def createValidationFailureException(self, exception: str):
        raise Exception(exception)

    def createConfigurationInvalidException(self, exception: str):
        raise Exception(exception)

    def test_omop_tables_hydrated_with_expected_counts(self):

        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")

        omop_lakehouse = None
        for lh in deployed_lakehouses.keys():
            if "omop" in str(lh).lower():
                omop_lakehouse = deployed_lakehouses[lh]

        omop_lakehouse_id = omop_lakehouse["id"]
        tables_path = f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{omop_lakehouse_id}/Tables"
        table_infos = mssparkutils.fs.ls(tables_path)
        lakehouse_display_name = omop_lakehouse["displayName"]
        
        for table_info in table_infos:
            table_name = table_info.path.split("/")[-1]
            print(table_name)
            row_count = spark.sql(f"SELECT COUNT(*) AS count FROM `{lakehouse_display_name}`.`{table_name}`").collect()[0]['count']
            
            self.assertTrue(EXPECTED_TABLE_COUNTS[table_name] == row_count)

    def test_person_table_has_location_ids_set(self):

        df = spark.sql("SELECT * FROM healthcare1_msft_gold_omop.person as person where person.location_id is NULL")
        self.assertTrue(df.rdd.isEmpty())

    def test_clinical_tables_have_person_id_and_date_modified(self):
        
        lakehouse_name = "healthcare1_msft_gold_omop"
        clinical_tables = [
            "observation", 
            "procedure_occurrence",
            "visit_occurrence",
            "drug_exposure"
        ]

        for table_name in clinical_tables:
            query = f"""
                SELECT
                    COUNT(*) AS total_rows,
                    COUNT(DISTINCT {table_name}_id) AS unique_ids,
                    COUNT({table_name}.msftModifiedDatetime) AS non_null_date_modified,
                    COUNT(person.person_id) AS valid_person_ids
                FROM {lakehouse_name}.{table_name}
                LEFT JOIN {lakehouse_name}.person AS person
                ON {table_name}.person_id = person.person_id
            """

            print(table_name)
            df = spark.sql(query)

            # Check if 'id' column has unique values and 'date_modified' column is not NULL
            result = df.collect()[0]
            total_rows = result['total_rows']
            unique_ids = result['unique_ids']
            non_null_date_modified = result['non_null_date_modified']

            self.assertTrue(total_rows == unique_ids)
            self.assertTrue(total_rows == non_null_date_modified)

    def test_concepts_are_mapped_to_vocabularies(self):
        lakehouse_name = "healthcare1_msft_gold_omop"
        concept_table = f"{lakehouse_name}.concept"
        vocab_table = f"{lakehouse_name}.vocabulary"

        # Query to check if each concept has a valid vocabulary_id
        query = f"""
        SELECT c.concept_id, c.vocabulary_id
        FROM {concept_table} AS c
        LEFT JOIN {vocab_table} AS v
        ON c.vocabulary_id = v.vocabulary_id
        WHERE v.vocabulary_id IS NULL and c.concept_id IS NOT NULL
        """

        # Execute the query
        df = spark.sql(query)

        df.show()

        # Display the result
        self.assertTrue(df.rdd.isEmpty())

    def test_validate_standard_concepts_hydration(self):
        
        query = """
        select c.concept_id, c.domain_id, c.standard_concept, count(t.care_site_id) as care_site_id_count
        from healthcare1_msft_gold_omop.care_site t
        join healthcare1_msft_gold_omop.concept c on t.place_of_service_concept_id = c.concept_id
        group by c.concept_id, c.domain_id, c.standard_concept  
        order by 1,3 desc
        """

        df = spark.sql(query)
        self.validate_df_cols_standard_nonstandard(df, 'care_site_id_count', 1)
        self.validate_df_cols_standard_nonstandard(df, None , 0, False)

    def test_validate_condition_occurrence_mapping(self):
        from pyspark.sql.functions import col
        query="""
        select  c.domain_id, c.standard_concept,  count(t.condition_occurrence_id) condition_occurence_count,
        round(count(t.condition_occurrence_id) * 100.0 / sum(count(t.condition_occurrence_id)) OVER(), 2) percent_of_total
        from healthcare1_msft_gold_omop.condition_occurrence t
        left join healthcare1_msft_gold_omop.concept c on t.condition_concept_id = c.concept_id
        where c.domain_id != 'Metadata'
        group by c.domain_id, c.standard_concept   
        order by 1,3 desc 
        """
        df = spark.sql(query)
        display(df)

        # -- atleast 1% are  mapped to standard concept in the condition domain
        self.validate_df_cols_standard_nonstandard(df, 'condition_occurence_count', 1)

        self.validate_df_cols_standard_nonstandard(df, None , 0, False)

    def test_validate_condition_occurrence_to_standard_concept_mapping(self):
        condition_ocurrence_to_standard_mapping_query = """
        select c.domain_id, c.standard_concept,  count(t.condition_occurrence_id) condition_occurrence_id_count,
        round(count(t.condition_occurrence_id) * 100.0 / sum(count(t.condition_occurrence_id)) OVER(), 2) percent_of_total
        from healthcare1_msft_gold_omop.condition_occurrence t
        left join healthcare1_msft_gold_omop.concept c on t.condition_status_concept_id = c.concept_id
        group by c.domain_id, c.standard_concept
        order by 1,3 desc
        """
        df = spark.sql(condition_ocurrence_to_standard_mapping_query)
        display(df)

        # assert 100% of 'Condition' domain are mapped to Standard_concept
        self.validate_df_cols_standard_nonstandard(df, 'condition_occurrence_id_count', 1)
        self.validate_df_cols_standard_nonstandard(df, None , 0, False)

    def test_validate_condition_occurrence_source_table_mapping(self):
        query = """
        SELECT sourcetable, count(*) as source_table_row_count,
        round(count(*) * 100.0 / sum(count(*)) OVER(), 2) percent_of_total
        from healthcare1_msft_gold_omop.condition_occurrence group by sourcetable
        """
        df = spark.sql(query)
        display(df)

        #  assert source table in condition_occurence has 100% condition_fhir
        self.assertTrue(df.collect()[0].__getitem__('percent_of_total') == 100)

    def test_measurement_ids_mapped_to_standard_concept(self):
        query="""
        select c.domain_id, c.standard_concept,  count(t.measurement_id) measurement_id_count,
        round(count(t.measurement_id) * 100.0 / sum(count(t.measurement_id)) OVER(), 2) percent_of_total
        from healthcare1_msft_gold_omop.measurement t
        left join healthcare1_msft_gold_omop.concept c on t.measurement_concept_id = c.concept_id
        group by c.domain_id, c.standard_concept   
        order by 1,3 desc
        """
        df = spark.sql(query)
        display(df)

        #  assert 100% of measurement_id are mapped to standard concept
        measurement_percentage = df.filter((col('domain_id') =='Measurement') & (col('standard_concept') == 'S')).collect()[0].__getitem__('percent_of_total')
        self.validate_df_col_count(measurement_percentage, 1)
        self.validate_df_cols_standard_nonstandard(df, None , 0, False)

    def test_location_table_has_unique_entries(self):
        lakehouse_name = "healthcare1_msft_gold_omop"
        table_name = "location"
        query = f"""
            SELECT
                COUNT(*) AS total_rows,
                COUNT(DISTINCT {table_name}_id) AS unique_ids
            FROM {lakehouse_name}.{table_name}
            """
        print(query)
        df = spark.sql(query).collect()[0]
        total_rows = df['total_rows']
        unique_ids = df['unique_ids']
        self.assertTrue(total_rows == unique_ids)
    

    def test_validate_measurement_concepts_mapped_to_standard(self):
        query = """
        select c.domain_id, c.standard_concept,  count(t.measurement_id) measurement_id_count,
        round(count(t.measurement_id) * 100.0 / sum(count(t.measurement_id)) OVER(), 2) percent_of_total
        from healthcare1_msft_gold_omop.measurement t
        left join healthcare1_msft_gold_omop.concept c on t.value_as_concept_id = c.concept_id
        where domain_id != 'Metadata'
        group by c.domain_id, c.standard_concept   
        order by 1,3 desc
        """
        df = spark.sql(query)
        display(df)
        self.validate_df_cols_standard_nonstandard(df, 'measurement_id_count', 1)

        condition = df.filter((col('domain_id') =='Condition') & (col('standard_concept') == 'S')).collect()[0].__getitem__('percent_of_total')
        observation = df.filter((col('domain_id') =='Observation') & (col('standard_concept') == 'S')).collect()[0].__getitem__('percent_of_total')

        self.validate_df_col_count(condition, 1)
        self.validate_df_col_count(observation, 1)
        self.validate_df_cols_standard_nonstandard(df, None , 0, False)

    def test_measurement_source_table_mapped_to_observation(self):
        query = """
        select SourceTable, Count(*) source_table_row_count,
        round(count(*) * 100.0 / sum(count(*)) OVER(), 2) percent_of_total
        from healthcare1_msft_gold_omop.measurement
        group by SourceTable
        """
        df = spark.sql(query)
        display(df)
        self.assertTrue(df.collect()[0].__getitem__('percent_of_total') == 100)

    def test_validate_person_columns(self):
        main_tables = ['person']
        for table in main_tables:
            df = spark.sql(f"SELECT * FROM healthcare1_msft_gold_omop.{table} LIMIT 5")
            display(df)
            primary_key = f'{table}_id'
            filtered_columns = [col for col in df.columns if col.endswith('_concept_id')]
            filtered_columns.append(primary_key)
            no_of_cols = 23
            table_source_value = f'{table}_source_value'
            self.validate_df(df, filtered_columns, no_of_cols, filtered_columns)

    def test_validate_omop_table_concepts(self):
        omop_tables_concept_pairs: Dict[str, list[int]] = {
            'person': ['gender_concept_id', 'race_concept_id', 'ethnicity_concept_id'], 
            'visit_occurrence': ['visit_concept_id', 'discharged_to_concept_id'], 
            'provider': ['gender_concept_id'], 
            'care_site': ['place_of_service_concept_id'],
            'condition_occurrence': ['condition_concept_id', 'condition_status_concept_id'],
            'drug_exposure': ['drug_concept_id'],
            'procedure_occurrence': ['procedure_concept_id'],
            'note_nlp': ['note_nlp_concept_id'],
            }

        self.omop_concept_validation(omop_tables_concept_pairs)

    def test_validate_omop_data_quality(self):
        
        deployed_lakehouses = self.get_workspace_items_by_type(self.workspace_id, "Lakehouse")
        silver_lakehouse = None
        for lh in deployed_lakehouses.keys():
            if "silver" in str(lh).lower():
                silver_lakehouse = deployed_lakehouses[lh]

        silver_to_omop_validation_config = {
            'Patient': [
                ('person', None, None),
                ('death', "isnotnull(deceasedDateTime)", None)
            ],
            'Encounter': [
                ('visit_occurrence', "subject.type = 'Patient'", None)
            ],
            'Observation': [
                ('observation', "subject.type = 'Patient' and category[0].coding[0].code != 'laboratory'", None),
                ('measurement', "subject.type = 'Patient' and category[0].coding[0].code == 'laboratory'", None)
            ],
            'Practitioner': [
                ('provider', None, None)
            ],
            'Organization': [
                ("care_site", None, None)
            ],
            'Condition': [
                ('condition_occurrence', "subject.type = 'Patient'", None)
            ],
            'MedicationRequest': [
                ('drug_exposure', "subject.type = 'Patient'", None)
            ],
            'DocumentReference': [
                ('note', "subject.type = 'Patient'", None)
            ],
            'Procedure': [
                ('procedure_occurrence', "subject.type = 'Patient' and isnull(focalDevice)", None),
                ('device_exposure', "subject.type = 'Patient' and isnotnull(focalDevice)", None)
            ],
        }

        # verify data transformation quality for all source tables
        self.verify_transformed_data_quality(silver_to_omop_validation_config, silver_lakehouse["displayName"], True)

def run_tests_and_write_output(spark):
    # Load and run tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(OmopDataValidationTests)

    # Inject parameters / context to the tests
    for test in suite:
        test.spark = spark
        test.workspace_id = workspace_id

    # Write XML test report to stream, decode after completion
    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream).run(suite)
    xml_output = write_stream.getvalue().decode('utf-8')

    # Write report to lakehouse
    mssparkutils.fs.put(f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)
    return xml_output

report = run_tests_and_write_output(spark)


StatementMeta(, 9dfa215f-5d98-43a2-80b9-6b00a1402949, 25, Finished, Available, Finished)


Running tests...
----------------------------------------------------------------------
  test_bronze_to_silver_data_ingestion (__main__.ClinicalFoundationsIngestionTests.test_bronze_to_silver_data_ingestion) ... 

Discrepancies found between ClinicalFHIR and Lakehouse tables:


SynapseWidget(Synapse.DataFrame, 0a75793e-e592-4057-ac24-e01134f2514b)

SynapseWidget(Synapse.DataFrame, 16ab4d73-b5fe-4da6-bb9c-b37b40d35380)

ok (67.382s)
  test_raw_to_bronze_data_ingestion (__main__.ClinicalFoundationsIngestionTests.test_raw_to_bronze_data_ingestion) ... 

Total number of records across all NDJSON files: 33316918
Total number of NDJSON files in sample data: 68

Resource Type Counts:
Goal: 100000
DocumentReference: 12814
DiagnosticReport: 43
MedicationRequest: 2675424
Observation: 19731849
RiskAssessment: 7
CarePlan: 41945
ExplanationOfBenefit: 40707
Procedure: 5299758
Patient: 57606
PractitionerRole: 15614
Organization: 15614
Location: 40937
Condition: 1973344
Encounter: 3195641
Practitioner: 15614
AllergyIntolerance: 1
Appointment: 100000
Discrepancies found between raw source NDJSON data and Bronze ClinicalFHIR table:


SynapseWidget(Synapse.DataFrame, 04b84bdb-4047-4c5d-b632-fd90033eee61)

ok (54.851s)

----------------------------------------------------------------------
Ran 2 tests in 122.235s

OK

Generating XML reports...
